# MNE-bids

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mne
from mne_bids import (
    BIDSPath,
    read_raw_bids,
)

plt.switch_backend('Qt5Agg')
plt.ion()

In [ ]:
import sys

sys.path.insert(0, str(Path.cwd().parent / 'code'))
from preprocessing import preprocess
from psd_lof import find_bad_channels_psd_lof

In [ ]:
dataset = "ds003969"
tasks = ['think1', 'think2','med1breath', 'med2']

# Create a BIDS root directory in the MNE dataset folder
root = Path('/home/jackhoyle')
bids_root = root / 'mne_data' / dataset
bids_root.mkdir(parents=True, exist_ok=True)

In [ ]:
subject = "007"
task = 'med2'

In [ ]:
raw: mne.io.edf.edf.RawEDF = read_raw_bids(
    BIDSPath(subject=subject, root=bids_root, task=task, datatype="eeg"),
    verbose=False
)
raw.load_data()
raw.set_montage("colin27_1020", on_missing="warn")
assert type(raw) == mne.io.edf.edf.RawBDF

In [ ]:
preprocess(
    raw,
    reference='EXG6',
    subsample_freq=256,
    high_thresh=None,
    low_thresh=1,
    pop_rejcont=True,
    bad_channel_variance_threshold=None
)

# outlier/bad channel detection strategies

In [ ]:
# built in raw waveform based outlier detection
mne.preprocessing.find_bad_channels_lof(raw, threshold=1.5)

In [ ]:
# power spectrum based outlier detection
find_bad_channels_psd_lof(raw, threshold=2)

In [ ]:
epochs = mne.make_fixed_length_epochs(raw, duration=3, preload=True)

In [ ]:
epochs.average().detrend().plot_joint()

In [ ]:
import autoreject

ar = autoreject.AutoReject(random_state=11, n_jobs=1, verbose=False)
ar.fit(epochs[:30])  # fit on a few epochs to save time
epochs_ar, reject_log = ar.transform(epochs, return_log=True)

In [ ]:
reject_log.plot('horizontal')

In [ ]:
epochs_ar.average().detrend().plot_joint()

In [ ]:
epochs.compute_psd().plot()

In [ ]:
epochs_ar.compute_psd().plot()

In [ ]:
find_bad_channels_psd_lof(raw, threshold=2, log_bands=True)

In [ ]:
psd.plot(picks="eeg", average=False)

In [ ]:
raw.plot()